In [16]:
import math,torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from torchvision.utils import save_image

device = "cuda" if torch.cuda.is_available() else "cpu"
T,BATCH,EPOCHS = 400,128,20

betas = torch.linspace(1e-4,0.02,T,device=device)
alphas = 1. - betas
acp = torch.cumprod(alphas,0)
sqrt_acp,sqrt_om = acp.sqrt(),(1. - acp).sqrt()
coef1,coef2 = (1. / alphas).sqrt(),betas / sqrt_om

def extract(a,t):
    return a[t].view(-1,1,1,1)

class SinPosEmb(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.dim = dim
    def forward(self,t):
        half = self.dim // 2
        freq = torch.exp(torch.arange(half,device=t.device) * -(math.log(1e4) / (half - 1)))
        emb = t[:,None].float() *freq
        return torch.cat((emb.sin(),emb.cos()),dim=-1)

class Block(nn.Module):
    def __init__(self,in_ch,out_ch,tdim):
        super().__init__()
        self.c1,self.c2 = nn.Conv2d(in_ch,out_ch,3,padding=1),nn.Conv2d(out_ch,out_ch,3,padding=1)
        self.n1,self.n2 = nn.GroupNorm(8,in_ch),nn.GroupNorm(8,out_ch)
        self.tmlp = nn.Linear(tdim,out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch,out_ch,1)
    def forward(self,x,t):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.tmlp(t)[:,:,None,None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)

class Attn(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.norm,self.qkv,self.proj = nn.GroupNorm(8,ch),nn.Conv2d(ch,3 * ch,1),nn.Conv2d(ch,ch,1)
    def forward(self,x):
        b,c,h,w = x.shape
        q,k,v = (m.flatten(2).transpose(1,2) for m in self.qkv(self.norm(x)).chunk(3,1))
        o = F.softmax(q @ k.transpose(-2,-1) / math.sqrt(c),dim=-1) @ v
        return x + self.proj(o.transpose(1,2).reshape(b,c,h,w))

class UNet(nn.Module):
    def __init__(self,ch=64,tdim=128):
        super().__init__()
        self.sin = SinPosEmb(tdim)
        self.tmlp = nn.Sequential(nn.Linear(tdim,tdim),nn.SiLU(),nn.Linear(tdim,tdim))
        self.inp = nn.Conv2d(1,ch,3,padding=1)
        self.d1,self.dn1 = Block(ch,ch,tdim),nn.Conv2d(ch,2 * ch,3,2,1)
        self.d2,self.dn2 = Block(2 * ch, 2 * ch,tdim),nn.Conv2d(2 * ch,2 * ch,3,2,1) 
        self.mid,self.attn = Block(2 * ch,2 * ch,tdim),Attn(2 * ch)
        self.up2,self.u2 = nn.ConvTranspose2d(2 * ch,2 * ch,4,2,1),Block(2 * ch,2 * ch,tdim)
        self.up1,self.u1 = nn.ConvTranspose2d(2 * ch,ch,4,2,1),Block(ch,ch,tdim)
        self.out = nn.Conv2d(ch,1,3,padding=1)
    def forward(self,x,t):
        t = self.tmlp(self.sin(t))
        h1 = self.d1(self.inp(x),t)
        h2 = self.d2(self.dn1(h1),t)
        h = self.attn(self.mid(self.dn2(h2),t))
        h = self.u2(self.up2(h),t) + h2
        h = self.u1(self.up1(h),t) + h1
        return self.out(h) 

model = UNet().to(device)
opt = torch.optim.AdamW(model.parameters(),lr=2e-4)
ds = datasets.MNIST("./data",train=True,download=False,transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize(0.5,0.5)]))
loader = DataLoader(ds,batch_size=BATCH,shuffle=True,drop_last=True)

def train():
    model.train()
    for ep in range(EPOCHS):
        for x,_ in loader:
            x = x.to(device)
            t = torch.randint(0,T,(x.size(0),),device=device)
            noise = torch.randn_like(x)
            x_t = extract(sqrt_acp,t) * x + extract(sqrt_om,t) * noise
            loss = F.mse_loss(model(x_t,t),noise)
            opt.zero_grad();loss.backward();opt.step()
        print(f"epoch {ep + 1}/{EPOCHS} loss {loss.item():.4f}")
@torch.no_grad()
def sample(model,n=64):
    model.eval()
    x = torch.randn(n,1,28,28,device=device)
    for t in reversed(range(T)):
        tb = torch.full((n,),t,device=device,dtype=torch.long)
        eps = model(x,tb)
        mean = extract(coef1,tb) * (x - extract(coef2,tb) * eps)
        x = mean + betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean
    return x
if __name__ == "__main__":
    train()
    save_image(sample(model),"ddpm_samples.png",nrow=8,normalize=True,value_range=(-1,1))

epoch 1/20 loss 0.0651
epoch 2/20 loss 0.0480
epoch 3/20 loss 0.0426
epoch 4/20 loss 0.0372
epoch 5/20 loss 0.0480
epoch 6/20 loss 0.0381
epoch 7/20 loss 0.0374
epoch 8/20 loss 0.0409
epoch 9/20 loss 0.0476
epoch 10/20 loss 0.0407
epoch 11/20 loss 0.0400
epoch 12/20 loss 0.0378


KeyboardInterrupt: 

In [4]:
import pandas as pd
df = pd.read_csv('data1/learn_pandas.csv')
gb = df.groupby(['School','Grade'])
print(gb.filter(lambda x:x.shape[0] > 10).head())

  School     Grade            Name  Gender  Height  Weight Transfer  \
0      A  Freshman    Gaopeng Yang  Female   158.9    46.0        N   
1      B  Freshman  Changqiang You    Male   166.5    70.0        N   
2      A    Senior         Mei Sun    Male   188.9    89.0        N   
5      D  Freshman     Xiaoli Qian  Female   158.0    51.0        N   
6      A  Freshman       Qiang Chu  Female   162.5    52.0        N   

   Test_Number   Test_Date Time_Record  
0            1   2019/10/5     0:04:34  
1            1    2019/9/4     0:04:20  
2            2   2019/9/12     0:05:22  
5            1  2019/10/31     0:03:47  
6            1  2019/12/12     0:03:53  


In [5]:
def BMI(x):
    Height = x['Height']/100
    Weight = x['Weight']
    BMI_value = Weight/Height ** 2
    return BMI_value.mean()
gb.apply(BMI)

School  Grade    
A       Freshman     21.208448
        Junior       19.429628
        Senior       21.451075
        Sophomore    19.772455
B       Freshman     21.225845
        Junior       19.674964
        Senior       21.577345
        Sophomore    17.611119
C       Freshman     19.969411
        Junior       20.115806
        Senior       19.573687
        Sophomore    21.712565
D       Freshman     18.465769
        Junior       21.352761
        Senior       20.189919
        Sophomore    20.458102
dtype: float64

In [10]:
gb = df.groupby(['Gender','Test_Number'])[['Height','Weight']]
print(gb.apply(lambda x:0))
print(gb.apply(lambda x:[0,0]))
gb.apply(lambda x:pd.Series([0,0],index=['a','b']))

Gender  Test_Number
Female  1              0
        2              0
        3              0
Male    1              0
        2              0
        3              0
dtype: int64
Gender  Test_Number
Female  1              [0, 0]
        2              [0, 0]
        3              [0, 0]
Male    1              [0, 0]
        2              [0, 0]
        3              [0, 0]
dtype: object


a  b
Gender Test_Number      
Female 1            0  0
       2            0  0
       3            0  0
Male   1            0  0
       2            0  0
       3            0  0

In [12]:
import numpy as np
gb.apply(lambda x:pd.DataFrame(np.ones((2,2)),
         index=['a','b'],
         columns=pd.Index([('w','x'),('y','z')])))


w    y
                        x    z
Gender Test_Number            
Female 1           a  1.0  1.0
                   b  1.0  1.0
       2           a  1.0  1.0
                   b  1.0  1.0
       3           a  1.0  1.0
                   b  1.0  1.0
Male   1           a  1.0  1.0
                   b  1.0  1.0
       2           a  1.0  1.0
                   b  1.0  1.0
       3           a  1.0  1.0
                   b  1.0  1.0

In [25]:
import pandas as pd
import numpy as np
print(pd.DataFrame({'Gender':['F','F','M','M'],'Height':[163,160,175,180]}))
print(pd.DataFrame({'Height: F':[163,160],'Height: M':[175,180]}))
df = pd.DataFrame({'Class':[1,1,2,2],
                   'Name':['San Zhang','San Zhang','Si Li','Si Li'],
                   'Subject':['Chinese','Math','Chinese','Math'],
                   'Grade':[80,75,90,85]})
df

  Gender  Height
0      F     163
1      F     160
2      M     175
3      M     180
   Height: F  Height: M
0        163        175
1        160        180


,Class,Name,Subject,Grade
0,1,San Zhang,Chinese,80
1,1,San Zhang,Math,75
2,2,Si Li,Chinese,90
3,2,Si Li,Math,85


In [30]:
df = pd.DataFrame({'Class':[1,1,2,2,1,1,2,2],
                   'Name':['San Zhang','San Zhang','Si Li','Si Li',
                           'San Zhang','San Zhang','Si Li','Si Li'],
                           'Examination':['Mid','Final','Mid','Final',
                                          'Mid','Final','Mid','Final'],
                           'Subject':['Chinese','Chinese','Chinese','Chinese',
                                      'Math','Math','Math','Math'],
                           'Grade':[80,75,85,65,90,85,92,88],
                           'rank':[10,15,21,15,20,7,6,2]})
print(df)
pivot_multi = df.pivot(index=['Class','Name'],
                       columns=['Subject','Examination'],
                       values=['Grade','rank'])
print(pivot_multi)

   Class       Name Examination  Subject  Grade  rank
0      1  San Zhang         Mid  Chinese     80    10
1      1  San Zhang       Final  Chinese     75    15
2      2      Si Li         Mid  Chinese     85    21
3      2      Si Li       Final  Chinese     65    15
4      1  San Zhang         Mid     Math     90    20
5      1  San Zhang       Final     Math     85     7
6      2      Si Li         Mid     Math     92     6
7      2      Si Li       Final     Math     88     2
                  Grade                     rank                 
Subject         Chinese       Math       Chinese       Math      
Examination         Mid Final  Mid Final     Mid Final  Mid Final
Class Name                                                       
1     San Zhang      80    75   90    85      10    15   20     7
2     Si Li          85    65   92    88      21    15    6     2
